In [11]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Note: you may need to restart the kernel to use updated packages.


### Read Data

In [12]:
def read_data():
    return [
        pd.read_csv('../data/development_data/awards_players.csv'),
        pd.read_csv('../data/development_data/coaches.csv'),
        pd.read_csv('../data/development_data/players.csv'),
        pd.read_csv('../data/development_data/players_teams.csv'),
        pd.read_csv('../data/development_data/series_post.csv'),
        pd.read_csv('../data/development_data/teams.csv'),
        pd.read_csv('../data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

In [13]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=[ 'college', 'collegeOther', 'deathDate'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series' too ??
teams = teams.drop(columns=['lgID', 'franchID', 'confID', 'divID', 'arena', 'name'])
teams_post = teams_post.drop(columns=['lgID'])



### Data Merging

In [14]:
#team metrics

data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

data['playoff_qualification'] = data['playoff'].apply(lambda x: 1.0 if x == 'Y' else 0.0)
data.drop(columns=['playoff'], inplace=True)
data.fillna({'W' : 0, 'L' : 0}, inplace=True)
    
#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum'
}).reset_index()

data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")


data.columns

#awards
# awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')

# data = pd.merge(data, awards_count, on=["year", "playerID"], how="left")
# data['num_awards'] = data['num_awards'].fillna(0)

Index(['year', 'tmID', 'rank', 'seeded', 'firstRound', 'semis', 'finals',
       'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm', 'o_3pa', 'o_oreb',
       'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to', 'o_blk', 'o_pts',
       'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa', 'd_oreb',
       'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk', 'd_pts',
       'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB', 'won_x',
       'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL',
       'min', 'attend', 'W', 'L', 'playoff_qualification', 'points',
       'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'won_y',
       'lost_y', 'post_wins', 'post_losses'],
      dtype='object')

In [15]:
data['win_loss_ratio'] = data['won_x'] / (data['won_x'] + data['lost_x'])  
data['avg_points_per_player'] = data['points'] / data['rank'] 
data['next_year_playoff_qualification'] = data.groupby('tmID')['playoff_qualification'].shift(-1)

columns_to_drop = ['tmID', 'lgID', 'rank', 'firstRound', 'semis', 'finals']
data = data.drop(columns=columns_to_drop, errors='ignore')

In [16]:
data.fillna(0, inplace=True)

### Model training

In [17]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier, Perceptron
from sklearn.metrics import accuracy_score

Note: you may need to restart the kernel to use updated packages.


#### Initialization 

In [18]:
#result lists
accuracy_scores = []
error_scores = []

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['next_year_playoff_qualification']]
target_column = 'next_year_playoff_qualification'
classes = np.unique(data[target_column])

#Create model
model = SGDClassifier(loss='log_loss', random_state=21)
#model = Perceptron(random_state=21)

#### Year training cycle

In [19]:
for year in sorted(data['year'].unique())[:-1]:  # Leave out the last year since it won't have a year after it for testing
    # Separate train and test data
    train_data = data[data['year'] == year]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip
    if test_data.empty:
        continue

    # Split features and target
    X_train = train_data[feature_columns]
    y_train = train_data[target_column]

    X_test = test_data[feature_columns]
    y_test = test_data[target_column]
    
    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train the model
    model.partial_fit(X_train, y_train, classes=classes)

    # Make predictions on the test set
    # y_pred_proba = model.predict_proba(X_test)[:,1]
    logit_scores = model.decision_function(X_test)
    y_pred_proba = 1 / (1 + np.exp(-logit_scores))  # Convert to probabilities
    y_pred_proba = np.round(y_pred_proba * 8 / sum(y_pred_proba),2)  # Normalize to sum to 8


    y_pred = np.zeros_like(y_pred_proba)
    indices = np.argsort(y_pred_proba)[-8:]  # Get indices of top 8 probabilities
    y_pred[indices] = 1                      # and set them to 1

    # Calculate accuracy and error
    accuracy_scores.append(round(accuracy_score(y_test, y_pred),2))

    error_array = np.abs(y_pred_proba - y_test.values)
    error_score = round(sum(error_array) / len(error_array),2)
    error_scores.append(error_score)

    # Output results for each year
    print(f"Year {year} -> {year + 1}:")
    print(f"Results: \n predict: {y_pred_proba}\n label: \t {y_pred}\n expected: {y_test.values}\n error: \t {error_array}")
    print(f"  Accuracy: {accuracy_scores[-1]}")
    print(f"  Error: \t {round(sum(error_array), 2)} / {len(error_array)} ({error_score})")
    print("\n")

Year 1 -> 2:
Results: 
 predict: [0.   1.14 0.   1.14 0.   1.14 1.14 0.   0.   1.14 0.   0.   1.14 0.
 1.14 0.  ]
 label: 	 [0. 1. 0. 1. 0. 1. 1. 0. 0. 1. 0. 0. 1. 0. 1. 1.]
 expected: [1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1.]
 error: 	 [1.   1.14 0.   0.14 1.   0.14 1.14 0.   1.   1.14 0.   0.   1.14 1.
 0.14 1.  ]
  Accuracy: 0.5
  Error: 	 9.98 / 16 (0.62)


Year 2 -> 3:
Results: 
 predict: [0.89 0.89 0.89 0.   0.   0.   0.89 0.89 0.   0.   0.89 0.89 0.89 0.
 0.89 0.  ]
 label: 	 [0. 1. 1. 0. 0. 0. 1. 1. 0. 0. 1. 1. 1. 0. 1. 0.]
 expected: [1. 1. 1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0.]
 error: 	 [0.11 0.11 0.11 1.   0.   1.   0.89 0.11 0.   0.   0.89 0.89 0.11 0.
 0.89 0.  ]
  Accuracy: 0.56
  Error: 	 6.11 / 16 (0.38)


Year 3 -> 4:
Results: 
 predict: [0.  1.6 0.  0.  1.6 0.  0.  0.  1.6 0.  0.  1.6 1.6 0. ]
 label: 	 [0. 1. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 1. 1.]
 expected: [0. 0. 1. 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1.]
 error: 	 [0.  1.6 1.  1.  1.6 0.  1.  1.  0.6 0.  1.  

/tmp/ipykernel_40221/3706253640.py:28: RuntimeWarning: overflow encountered in exp
  y_pred_proba = 1 / (1 + np.exp(-logit_scores))  # Convert to probabilities
/tmp/ipykernel_40221/3706253640.py:28: RuntimeWarning: overflow encountered in exp
  y_pred_proba = 1 / (1 + np.exp(-logit_scores))  # Convert to probabilities
/tmp/ipykernel_40221/3706253640.py:28: RuntimeWarning: overflow encountered in exp
  y_pred_proba = 1 / (1 + np.exp(-logit_scores))  # Convert to probabilities
/tmp/ipykernel_40221/3706253640.py:28: RuntimeWarning: overflow encountered in exp
  y_pred_proba = 1 / (1 + np.exp(-logit_scores))  # Convert to probabilities


### End Results

In [20]:
print(f"Accuracy  {accuracy_scores}")
print(f"Error \t {[float(e) for e in error_scores]}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores[:-1]) / len(accuracy_scores[:-1]):.2f}")
print(f"  Average Error: \t {round(sum(error_scores[:-1]) / len(error_scores[:-1]),2)}")

Accuracy  [0.5, 0.56, 0.43, 0.54, 0.69, 0.71, 0.23, 0.43, 0.38]
Error 	 [0.62, 0.38, 0.86, 0.62, 0.27, 0.38, 0.92, 0.57, 0.61]

Average Performance Over All Years:
  Average Accuracy: 0.51
  Average Error: 	 0.58
